In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
import k3d
import sys
import os

import trimesh
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import PlaceholderNet, GeneralNet
from training.residuals import (bind_model, grad_x_f, hess_x_f, r_data, grad_theta_r_data, r_normal, grad_theta_r_normal, r_laplacian, grad_theta_r_laplacian,
r_mean_curvature, grad_theta_r_mean_curvature, r_gauss_curvature, grad_theta_r_gauss_curvature, r_eikonal, grad_theta_r_eikonal, r_principle_curvature_1, grad_theta_r_principle_curvature_1)
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cpu' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

In [2]:
# mesh = trimesh.load("bun_zipper.ply")
mesh = trimesh.load("rocker-arm.off")
pts = torch.tensor(mesh.vertices, dtype=torch.float64)
center = pts.mean(dim=0)
pts -= center
scale = pts.norm(dim=1).max()
pts /= scale
model = PlaceholderNet(ks=[3, 32, 32, 32, 1], act=torch.sin)
model.load_state_dict(torch.load('3,32,32,32,1,sin,rockerarm,gn', map_location=torch.device('cpu')))
model.double()

PlaceholderNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [3]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=pts.min(dim=0).values-0.01,
    bbox_max=pts.max(dim=0).values+0.15,
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig.display()
model.double()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

PlaceholderNet(
  (fcs): ModuleList(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=True)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [9]:
pts = pts[:100]
bind_model(model)

In [10]:
loss_weights = {"data": 1.0, "eikonal": 0.01, "normal": 1.0}
config = {
    "pts_data": pts,
    "pts_eikonal": pts,
    "pts_surface": pts,
    "true_normals": pts,
    "loss_weights": loss_weights,
    "regularization": 1e-6,    
}

In [11]:
def make_residual_fn(config):
    loss_weights = config["loss_weights"]

    def residual_fn(params):
        r_blocks = []

        # Data
        if "pts_data" in config and loss_weights.get("data", 0.0) > 0:
            pts = config["pts_data"]
            N = pts.shape[0]
            vals = config.get("vals_data", torch.zeros(N, dtype=pts.dtype))
            r = r_data(params, pts, vals).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["data"] / N, dtype=r.dtype)) * r
            )

        # Eikonal
        if "pts_eikonal" in config and loss_weights.get("eikonal", 0.0) > 0:
            pts = config["pts_eikonal"]
            N = pts.shape[0]
            r = r_eikonal(params, pts).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["eikonal"] / N, dtype=r.dtype)) * r
            )
            

        # Normal
        if "pts_surface" in config and loss_weights.get("normal", 0.0) > 0:
            pts = config["pts_surface"]
            N = pts.shape[0]
            true_normals = config["true_normals"]
            r = r_normal(params, pts, true_normals).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["normal"] / N, dtype=r.dtype)) * r
            )
            

        # Mean Curvature
        if "pts_surface" in config and loss_weights.get("mean_curvature", 0.0) > 0:
            pts = config["pts_surface"]
            N = pts.shape[0]
            r = r_mean_curvature(params, pts).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["mean_curvature"] / N, dtype=r.dtype)) * r
            )
            

        # Gauss Curvature
        if "pts_surface" in config and loss_weights.get("gauss_curvature", 0.0) > 0:
            pts = config["pts_surface"]
            N = pts.shape[0]
            r = r_gauss_curvature(params, pts).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["gauss_curvature"] / N, dtype=r.dtype)) * r
            )
            

        # Laplacian
        if "pts_surface" in config and loss_weights.get("laplacian", 0.0) > 0:
            pts = config["pts_surface"]
            N = pts.shape[0]
            r = r_laplacian(params, pts).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["laplacian"] / N, dtype=r.dtype)) * r
            )

        # Surface Strain (principle curvatures 1 and 2)
        if "pts_surface" in config and loss_weights.get("surface_strain", 0.0) > 0:
            pts = config["pts_surface"]
            N = pts.shape[0]
            r = r_principle_curvature_1(params, pts).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["surface_strain"] / N, dtype=r.dtype)) * r
            )
            r = r_principle_curvature_2(params, pts).squeeze(1)
            r_blocks.append(
                torch.sqrt(torch.tensor(loss_weights["surface_strain"] / N, dtype=r.dtype)) * r
            )

        r = torch.cat(r_blocks, dim=0)

        return r

    return residual_fn


In [12]:
residual_fn = make_residual_fn(config)

In [13]:
residual_fn(model.params).shape

torch.Size([100])
torch.Size([100])
torch.Size([100])


torch.Size([300])

In [31]:
from torch.func import vmap, jacrev, jacfwd, functional_call, jvp

In [20]:
jacobian_dict = jacrev(residual_fn)(model.params)

torch.Size([100])
torch.Size([100])
torch.Size([100])


In [29]:
J = torch.cat([p.flatten(start_dim=1) for p in jacobian_dict.values()], dim=1).T

In [41]:
# Ensure primals is a tuple containing model.params
primals = (model.params,)

# Compute the tangents as residual_fn(model.params) - ensure the return shape is correct
tangents = residual_fn(model.params)  # tangents is the output of residual_fn(model.params)

# Ensure tangents is a dictionary (as residual_fn(model.params) might return a tensor)
tangents = {key: value for key, value in zip(model.params.keys(), tangents)}

# If necessary, reshape or align tangents with primals, ensuring their shapes match
# For instance, if tangents need to be reshaped to match the shape of model parameters, do it here
# Example: if residual_fn outputs tensors of shape [32, 3], but the model has parameters with shape [32, 3], you may need to reshape accordingly.

# Wrap tangents in a tuple to match the structure of primals
tangents = (tangents,)

# Compute the Jacobian-vector product (JVP)
jvp = torch.func.jvp(residual_fn, primals, tangents)

# Now `jvp` contains the Jacobian-vector product of the residual function with respect to model.params


torch.Size([100])
torch.Size([100])
torch.Size([100])


RuntimeError: Trying to set a forward gradient that has a different size than that of the original Tensor, this is not supported. Tensor is of size [32, 3] while the given forward gradient is of size [].